# EmoViS Decisive Ablation Experiments

Two publication-grade ablation experiments for Vietnamese multi-label emotion recognition (ViGoEmotions, 28 classes).

| System | Description |
|--------|-------------|
| **A0 (Baseline)** | ViSoBERT + Weighted BCE + normalized text (emoji -> Vietnamese text) |
| **Exp1** | A0 + Emoji2Vec branch (isolates emoji-branch effect from loss change) |
| **Exp2** | A0 with raw Unicode emoji kept (tests if ViSoBERT tokenizer handles emoji natively) |

---

## SETUP INSTRUCTIONS

### Step 1: Create Kaggle Datasets (one-time)

You need to upload **2 datasets** to Kaggle before running this notebook:

#### Dataset 1: `vigoemotions`
Go to https://www.kaggle.com/datasets/create -> New Dataset  
Upload these files into it:
- `data/vigoemotions/train.csv`
- `data/vigoemotions/val.csv`
- `data/vigoemotions/test.csv`

Name the dataset: **vigoemotions**

#### Dataset 2: `emoji2vec-bin`
Go to https://www.kaggle.com/datasets/create -> New Dataset  
Upload:
- `data/emoji2vec.bin`

Name the dataset: **emoji2vec-bin**

### Step 2: Attach datasets to this notebook
In the Kaggle notebook editor, click **Add Data** (right sidebar) and add both:
- `vigoemotions`  
- `emoji2vec-bin`

### Step 3: Enable GPU
Settings -> Accelerator -> **GPU T4 x2**

### Step 4: Enable Internet
Settings -> Internet -> **On**

### Step 5: Run All Cells

---
## Cell 1: Verify GPU and Install Dependencies

In [ ]:
# -- Check GPU availability --
!nvidia-smi

# -- Install/upgrade dependencies --
!pip install -q gensim emoji transformers==4.52.4 scikit-learn

# -- Print versions for reproducibility log --
import torch, transformers, sklearn, numpy as np, pandas as pd, sys, platform
print(f"Python:       {sys.version}")
print(f"PyTorch:      {torch.__version__}")
print(f"Transformers: {transformers.__version__}")
print(f"scikit-learn: {sklearn.__version__}")
print(f"NumPy:        {np.__version__}")
print(f"Pandas:       {pd.__version__}")
try:
    import gensim; print(f"Gensim:       {gensim.__version__}")
except: print("Gensim: NOT INSTALLED")
try:
    import emoji; print(f"emoji:        {emoji.__version__}")
except: print("emoji: NOT INSTALLED")
print(f"CUDA:         {torch.version.cuda}")
print(f"GPU:          {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE'}")
print(f"GPU count:    {torch.cuda.device_count()}")

---
## Cell 2: Clone Repository

In [ ]:
import os

REPO_DIR = "/kaggle/working/repo"

if not os.path.exists(REPO_DIR):
    !git clone https://github.com/FlynnBui399/vietnamese-emoji-emotion-recognition.git {REPO_DIR}
    print(f"Cloned to {REPO_DIR}")
else:
    print(f"Repo already exists at {REPO_DIR}, pulling latest...")
    !cd {REPO_DIR} && git pull

# Show the script exists
script_path = os.path.join(REPO_DIR, "scripts", "run_decisive_ablations.py")
assert os.path.exists(script_path), f"Script not found at {script_path}"
print(f"Script found: {script_path}")

# Show git commit hash for reproducibility
!cd {REPO_DIR} && git log -1 --oneline

---
## Cell 3: Verify Data Paths

This cell checks that all required files are accessible. If any check fails, you need to attach the corresponding Kaggle Dataset.

In [ ]:
import os
from pathlib import Path

# -- Auto-detect data paths --
# Kaggle datasets are mounted under /kaggle/input/<dataset-name>/
# The exact subfolder structure depends on how you uploaded.

def find_file(name, search_dirs):
    """Search for a file in multiple possible locations."""
    for d in search_dirs:
        for root, dirs, files in os.walk(d):
            if name in files:
                return os.path.join(root, name)
    return None

KAGGLE_INPUT = "/kaggle/input"
REPO_DIR = "/kaggle/working/repo"

# Find train.csv to determine DATA_DIR
train_csv = find_file("train.csv", [KAGGLE_INPUT])
if train_csv is None:
    raise FileNotFoundError(
        "train.csv not found under /kaggle/input/. "
        "Please add the 'vigoemotions' dataset via Add Data in the sidebar."
    )
DATA_DIR = os.path.dirname(train_csv)
print(f"DATA_DIR: {DATA_DIR}")

# Verify all CSV splits exist
for split in ["train.csv", "val.csv", "test.csv"]:
    p = os.path.join(DATA_DIR, split)
    assert os.path.exists(p), f"Missing: {p}"
    print(f"  Found: {p}")

# Find emoji2vec.bin
emoji2vec_path = find_file("emoji2vec.bin", [KAGGLE_INPUT])
if emoji2vec_path is None:
    # Also check if it's in the repo's data/ folder
    emoji2vec_path = find_file("emoji2vec.bin", [REPO_DIR])
if emoji2vec_path is None:
    raise FileNotFoundError(
        "emoji2vec.bin not found. "
        "Please add the 'emoji2vec-bin' dataset via Add Data in the sidebar, "
        "or run: python scripts/download_emoji2vec.py"
    )
EMOJI2VEC_PATH = emoji2vec_path
print(f"EMOJI2VEC_PATH: {EMOJI2VEC_PATH}")

# docs/ directory is in the cloned repo
DOCS_DIR = os.path.join(REPO_DIR, "docs")
assert os.path.isdir(DOCS_DIR), f"docs/ not found at {DOCS_DIR}"
for f in ["patterns.json", "teencode4.txt", "emojis.json"]:
    assert os.path.exists(os.path.join(DOCS_DIR, f)), f"Missing: {DOCS_DIR}/{f}"
print(f"DOCS_DIR: {DOCS_DIR}")

OUTPUT_DIR = "/kaggle/working/emovis_decisive_experiments"
print(f"OUTPUT_DIR: {OUTPUT_DIR}")

print("\nAll data paths verified!")

---
## Cell 4: Run ALL Experiments (A0 + Exp1 + Exp2)

This runs all 9 training runs sequentially (3 systems x 3 seeds).  
**Estimated time: ~3-5 hours on T4 GPU.**

If Kaggle times out, just **re-run this cell** -- completed seeds are automatically skipped (resume logic).

In [ ]:
REPO_DIR = "/kaggle/working/repo"

!cd {REPO_DIR} && python scripts/run_decisive_ablations.py \
    --data_dir "{DATA_DIR}" \
    --docs_dir "{DOCS_DIR}" \
    --emoji2vec_path "{EMOJI2VEC_PATH}" \
    --output_dir "{OUTPUT_DIR}" \
    --model_name uitnlp/visobert \
    --max_length 160 \
    --batch_size 32 \
    --eval_batch_size 64 \
    --epochs 12 \
    --patience 3 \
    --dropout 0.2 \
    --lr 5e-5 \
    --weight_decay 0.01 \
    --warmup_epochs 1.0 \
    --num_workers 2 \
    --seeds 42,1,7

---
## (ALTERNATIVE) Cell 4a/4b/4c: Run Experiments Separately

If you prefer to run each experiment in separate Kaggle sessions (or if time is tight), use these cells instead of Cell 4.

### Cell 4a: Run ONLY A0 Baseline

In [ ]:
# UNCOMMENT TO RUN:
# !cd {REPO_DIR} && python scripts/run_decisive_ablations.py \
#     --data_dir "{DATA_DIR}" \
#     --docs_dir "{DOCS_DIR}" \
#     --emoji2vec_path "{EMOJI2VEC_PATH}" \
#     --output_dir "{OUTPUT_DIR}" \
#     --model_name uitnlp/visobert \
#     --max_length 160 --batch_size 32 --eval_batch_size 64 \
#     --epochs 12 --patience 3 --dropout 0.2 --lr 5e-5 \
#     --seeds 42,1,7 \
#     --skip_exp1 --skip_exp2

### Cell 4b: Run ONLY Experiment 1 (Emoji Branch)

In [ ]:
# UNCOMMENT TO RUN:
# !cd {REPO_DIR} && python scripts/run_decisive_ablations.py \
#     --data_dir "{DATA_DIR}" \
#     --docs_dir "{DOCS_DIR}" \
#     --emoji2vec_path "{EMOJI2VEC_PATH}" \
#     --output_dir "{OUTPUT_DIR}" \
#     --model_name uitnlp/visobert \
#     --max_length 160 --batch_size 32 --eval_batch_size 64 \
#     --epochs 12 --patience 3 --dropout 0.2 --lr 5e-5 \
#     --seeds 42,1,7 \
#     --skip_a0 --skip_exp2

### Cell 4c: Run ONLY Experiment 2 (Raw Emoji)

In [ ]:
# UNCOMMENT TO RUN:
# !cd {REPO_DIR} && python scripts/run_decisive_ablations.py \
#     --data_dir "{DATA_DIR}" \
#     --docs_dir "{DOCS_DIR}" \
#     --emoji2vec_path "{EMOJI2VEC_PATH}" \
#     --output_dir "{OUTPUT_DIR}" \
#     --model_name uitnlp/visobert \
#     --max_length 160 --batch_size 32 --eval_batch_size 64 \
#     --epochs 12 --patience 3 --dropout 0.2 --lr 5e-5 \
#     --seeds 42,1,7 \
#     --skip_a0 --skip_exp1

---
## Cell 5: Inspect Results

In [ ]:
import json
import pandas as pd
from pathlib import Path

OUTPUT_DIR = Path("/kaggle/working/emovis_decisive_experiments")

# -- Final Comparison Table --
table_path = OUTPUT_DIR / "final_comparison_table.csv"
if table_path.exists():
    df = pd.read_csv(table_path)
    print("=" * 100)
    print("FINAL CROSS-EXPERIMENT COMPARISON TABLE")
    print("=" * 100)
    display(df)
else:
    print(f"Table not found at {table_path}. Training may not be complete yet.")

print()

# -- Bootstrap Results --
bs_path = OUTPUT_DIR / "bootstrap_results.json"
if bs_path.exists():
    with open(bs_path) as f:
        bs = json.load(f)
    print("=" * 100)
    print("BOOTSTRAP SIGNIFICANCE RESULTS")
    print("=" * 100)
    for key, val in bs.items():
        print(f"\n{key}:")
        for k, v in val.items():
            print(f"  {k}: {v}")

print()

# -- Per-seed metrics --
for exp in ["a0_baseline_wbce", "exp1_emoji_branch_wbce", "exp2_raw_emoji_no_branch"]:
    exp_dir = OUTPUT_DIR / exp
    if not exp_dir.exists():
        continue
    print(f"\n{'='*60}")
    print(f"  {exp}")
    print(f"{'='*60}")
    for seed in [42, 1, 7]:
        m_path = exp_dir / f"seed_{seed}" / "metrics.json"
        if m_path.exists():
            with open(m_path) as f:
                m = json.load(f)
            t = m.get("test", {})
            print(f"  Seed {seed:>2}: MF1-opt={t.get('mf1_opt','-'):.4f}  "
                  f"MF1-fix={t.get('mf1_fix','-'):.4f}  "
                  f"MicroF1={t.get('micro_f1','-'):.4f}  "
                  f"mAP={t.get('map','-'):.4f}  "
                  f"(best_epoch={m.get('best_epoch','-')})")

---
## Cell 6: Emoji Tokenizer Diagnostic (Exp2)

In [ ]:
import json
from pathlib import Path

diag_path = Path("/kaggle/working/emovis_decisive_experiments/exp2_raw_emoji_no_branch/emoji_tokenizer_diagnostic.json")
if diag_path.exists():
    with open(diag_path) as f:
        diag = json.load(f)
    print("Emoji Tokenizer Diagnostic (Exp2)")
    print("="*50)
    for k, v in diag.items():
        print(f"  {k}: {v}")
    print()
    if diag.get("unk_rate", 1) < 0.1:
        print("CONCLUSION: ViSoBERT tokenizer handles raw emoji well (low UNK rate).")
    else:
        print("CONCLUSION: ViSoBERT tokenizer struggles with raw emoji (high UNK rate).")
else:
    print("Diagnostic not found. Run Exp2 first.")

---
## Cell 7: List All Output Artifacts

In [ ]:
import os
from pathlib import Path

OUTPUT_DIR = Path("/kaggle/working/emovis_decisive_experiments")
if OUTPUT_DIR.exists():
    total_size = 0
    for root, dirs, files in os.walk(OUTPUT_DIR):
        for f in sorted(files):
            fp = Path(root) / f
            size = fp.stat().st_size
            total_size += size
            rel = fp.relative_to(OUTPUT_DIR)
            print(f"  {rel}  ({size/1024:.1f} KB)")
    print(f"\nTotal: {total_size/1024/1024:.1f} MB")
else:
    print("Output directory not found. Run training first.")

---
## Cell 8: Download Artifacts

The zip file will appear in the Kaggle **Output** tab for download.

In [ ]:
import os
from pathlib import Path

OUTPUT_DIR = Path("/kaggle/working/emovis_decisive_experiments")
zip_path = OUTPUT_DIR / "emovis_decisive_experiments.zip"

if zip_path.exists():
    size_mb = zip_path.stat().st_size / 1024 / 1024
    print(f"Zip file ready: {zip_path} ({size_mb:.1f} MB)")
    print("Go to the Output tab to download.")
else:
    print("Zip file not found. Training may not be complete.")
    print("If training finished but zip is missing, manually zip:")
    print(f"  !cd /kaggle/working && zip -r emovis_results.zip emovis_decisive_experiments/")